In [0]:
import pyspark.sql.functions as f

In [0]:
def merge_write_bronze():
    container_path = "abfss://files-copied@dataretailsalesdashboard.dfs.core.windows.net/Bronze/"
    america_df.write.format("delta").mode("overwrite").save(container_path + "america_sales")
    india_df.write.format("delta").mode("overwrite").save(container_path + "india_sales")
    uae_df.write.format("delta").mode("overwrite").save(container_path + "uae_sales")

In [0]:
def init_write_bronze():
    container_path = "abfss://files-copied@dataretailsalesdashboard.dfs.core.windows.net/Bronze/"
    app_america_df.write.format("delta").mode("overwrite").save(container_path + "america_sales")
    app_india_df.write.format("delta").mode("overwrite").save(container_path + "india_sales")
    app_uae_df.write.format("delta").mode("overwrite").save(container_path + "uae_sales")
    

In [0]:
def sql_merge():
    global america_df, india_df, uae_df
    spark.sql("""
        MERGE INTO india_df AS t
        USING app_india_df AS s
        ON t.Transaction_ID = s.Transaction_ID
        WHEN MATCHED THEN UPDATE SET
            t.Customer_Name = s.Customer_Name,
            t.City = s.City,
            t.State = s.State,
            t.Country = s.Country,
            t.Product_Category = s.Product_Category,
            t.Product_Name = s.Product_Name,
            t.Quantity = s.Quantity,
            t.Unit_Price = s.Unit_Price,
            t.Total_Sales = s.Total_Sales,
            t.Payment_Method = s.Payment_Method,
            t.Order_Date = s.Order_Date,
            t.Customer_Email = s.Customer_Email,
            t.Store_ID = s.Store_ID,
            t.Date_Processed = s.Date_Processed
        WHEN NOT MATCHED THEN INSERT (
            Transaction_ID, Customer_Name, City, State, Country,
            Product_Category, Product_Name, Quantity, Unit_Price,
            Total_Sales, Payment_Method, Order_Date, Customer_Email,
            Store_ID, Date_Processed)
        VALUES (
            s.Transaction_ID, s.Customer_Name, s.City, s.State, s.Country,
            s.Product_Category, s.Product_Name, s.Quantity, s.Unit_Price,
            s.Total_Sales, s.Payment_Method, s.Order_Date, s.Customer_Email,
            s.Store_ID, s.Date_Processed)
    """)

    spark.sql("""
        MERGE INTO america_df AS t
        USING app_america_df AS s
        ON t.Transaction_ID = s.Transaction_ID
        WHEN MATCHED THEN UPDATE SET
            t.Customer_Name = s.Customer_Name,
            t.City = s.City,
            t.State = s.State,
            t.Country = s.Country,
            t.Product_Category = s.Product_Category,
            t.Product_Name = s.Product_Name,
            t.Quantity = s.Quantity,
            t.Unit_Price = s.Unit_Price,
            t.Total_Sales = s.Total_Sales,
            t.Payment_Method = s.Payment_Method,
            t.Order_Date = s.Order_Date,
            t.Customer_Email = s.Customer_Email,
            t.Store_ID = s.Store_ID,
            t.Date_Processed = s.Date_Processed
        WHEN NOT MATCHED THEN INSERT (
            Transaction_ID, Customer_Name, City, State, Country,
            Product_Category, Product_Name, Quantity, Unit_Price,
            Total_Sales, Payment_Method, Order_Date, Customer_Email,
            Store_ID, Date_Processed)
        VALUES (
            s.Transaction_ID, s.Customer_Name, s.City, s.State, s.Country,
            s.Product_Category, s.Product_Name, s.Quantity, s.Unit_Price,
            s.Total_Sales, s.Payment_Method, s.Order_Date, s.Customer_Email,
            s.Store_ID, s.Date_Processed)
    """)

    spark.sql("""
        MERGE INTO uae_df AS t
        USING app_uae_df AS s
        ON t.Transaction_ID = s.Transaction_ID
        WHEN MATCHED THEN UPDATE SET
            t.Customer_Name = s.Customer_Name,
            t.City = s.City,
            t.State = s.State,
            t.Country = s.Country,
            t.Product_Category = s.Product_Category,
            t.Product_Name = s.Product_Name,
            t.Quantity = s.Quantity,
            t.Unit_Price = s.Unit_Price,
            t.Total_Sales = s.Total_Sales,
            t.Payment_Method = s.Payment_Method,
            t.Order_Date = s.Order_Date,
            t.Customer_Email = s.Customer_Email,
            t.Store_ID = s.Store_ID,
            t.Date_Processed = s.Date_Processed
        WHEN NOT MATCHED THEN INSERT (
            Transaction_ID, Customer_Name, City, State, Country,
            Product_Category, Product_Name, Quantity, Unit_Price,
            Total_Sales, Payment_Method, Order_Date, Customer_Email,
            Store_ID, Date_Processed)
        VALUES (
            s.Transaction_ID, s.Customer_Name, s.City, s.State, s.Country,
            s.Product_Category, s.Product_Name, s.Quantity, s.Unit_Price,
            s.Total_Sales, s.Payment_Method, s.Order_Date, s.Customer_Email,
            s.Store_ID, s.Date_Processed)
    """)

    america_df = spark.table("america_df")
    india_df   = spark.table("india_df")
    uae_df     = spark.table("uae_df")

In [0]:

container_path = "abfss://files-copied@dataretailsalesdashboard.dfs.core.windows.net/source/"
contents_df = (dbutils.fs.ls(container_path))
file_names =[]
for contents in contents_df:
    file_names.append(contents.name)


app_india_df = spark.read.csv(container_path+file_names[1], header=True)
app_america_df = spark.read.csv(container_path+file_names[0], header=True)
app_uae_df = spark.read.csv(container_path+file_names[2], header=True)
app_india_df = app_india_df.withColumn("Date_processed", f.current_timestamp())
app_america_df = app_america_df.withColumn("Date_processed", f.current_timestamp())
app_uae_df = app_uae_df.withColumn("Date_processed", f.current_timestamp())
display(app_ame_df)

container_path = "abfss://files-copied@dataretailsalesdashboard.dfs.core.windows.net/Bronze/"
contents_df = []
contents_df = (dbutils.fs.ls(container_path))
file_names =[]
for contents in contents_df:
    file_names.append(contents.name)

if len(file_names) < 3 :
    init_write_bronze()
else:
    container_path = "abfss://files-copied@dataretailsalesdashboard.dfs.core.windows.net/Bronze/"
    india_df = spark.read.format("delta").load(container_path + "india_sales")
    america_df = spark.read.format("delta").load(container_path + "america_sales")
    uae_df = spark.read.format("delta").load(container_path + "uae_sales")
    india_df.createOrReplaceTempView("india_df")
    america_df.createOrReplaceTempView("america_df")
    uae_df.createOrReplaceTempView("uae_df")
    app_india_df.createOrReplaceTempView("app_india_df")
    app_america_df.createOrReplaceTempView("app_america_df")
    app_uae_df.createOrReplaceTempView("app_uae_df")
    sql_merge()
    merge_write_bronze()
